# RG3 역사 정보 가용성 감사

고정620개/가스19에서 past10 std4 및 noise3의 실제 NaN 여부를 현재 센서 전처리로 재확인. 기존 모델의 정상 중앙값 대체는 역사부족 가스도 정상같은 숫자로 만들 수 있다. 이것은 원인 가설이며 실제 실패 원인이라는 증거가 아님. 다음 대조에서 학습역사전문가는 완전한 기록만 학습하되 평가코호트19개를 그대로 유지하고 불완전 샷은 현재값 전문가로 다루는 방법을 검토.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
assert len(d)==620 and np.array_equal(y,z['labels']) and np.array_equal(d.TimeStamp.astype(str),z['timestamps'])
cols=[v+'__past10_std' for v in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
a=d[['EQUIP_CD','PART_NAME','TimeStamp']].copy()
a['actual_gas']=y;a['block']=groups.to_numpy()
a['valid_std4']=np.isfinite(d[cols].to_numpy(float)).all(1)
a['valid_noise3']=np.isfinite(d[[cols[0],cols[2],cols[3]]].to_numpy(float)).all(1)
a['missing_std_count']=d[cols].isna().sum(1)
for col in cols:a[col]=d[col]
a.to_csv(t/'rg3_history_availability_all620.csv',index=False,encoding='utf-8-sig')
summary=a.groupby(['actual_gas','valid_std4','valid_noise3']).size().reset_index(name='rows')
summary.to_csv(t/'rg3_history_availability_summary.csv',index=False,encoding='utf-8-sig')
print(summary.to_string(index=False))
print(a[(a.actual_gas==1)&(~a.valid_noise3)][['TimeStamp','block','missing_std_count']].to_string(index=False))
print('This identifies missingness, not proof that imputation caused earlier failures. No evaluation rows removed.')


 actual_gas  valid_std4  valid_noise3  rows
          0       False         False    10
          0        True          True   591
          1       False         False     1
          1        True          True    18
          TimeStamp  block  missing_std_count
2020-10-23 01:03:33      4                  4
This identifies missingness, not proof that imputation caused earlier failures. No evaluation rows removed.


C:\Users\hayeon\AppData\Local\Temp\ipykernel_18508\848164660.py:15: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  a['missing_std_count']=d[cols].isna().sum(1)
